In [1]:
"""
Примеры кода к Лекции 5.3: Агент с инструментами — tools, streaming, Command

Этот модуль демонстрирует:
1. Инструменты в LangGraph: bind_tools, ToolNode, ReAct-агент, create_agent
2. Command — маршрутизация изнутри узла
3. Streaming — потоковая обработка (5 режимов + v2)
4. Продвинутые паттерны: кастомный tool node, recursion_limit, фильтрация
"""

'\nПримеры кода к Лекции 5.3: Агент с инструментами — tools, streaming, Command\n\nЭтот модуль демонстрирует:\n1. Инструменты в LangGraph: bind_tools, ToolNode, ReAct-агент, create_agent\n2. Command — маршрутизация изнутри узла\n3. Streaming — потоковая обработка (5 режимов + v2)\n4. Продвинутые паттерны: кастомный tool node, recursion_limit, фильтрация\n'

In [2]:
import json
import operator
from typing import Annotated, Literal, TypedDict

from langchain.agents import create_agent
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.types import Command
from llm_config import check_api_key, get_llm

if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Инструменты в LangGraph

In [3]:
@tool
def multiply(a: float, b: float) -> float:
    """Умножает два числа."""
    return a * b


@tool
def web_search(query: str) -> str:
    """Ищет информацию в интернете."""
    # В реальности — вызов Tavily, Serper или другого API
    return f"Результаты по запросу '{query}': Python создан Гвидо ван Россумом в 1991 году."


@tool
def get_weather(city: str) -> str:
    """Возвращает текущую погоду в городе."""
    return f"В {city} сейчас +15°C, облачно."


# Общий список инструментов, используемый в нескольких примерах
tools = [multiply, web_search, get_weather]


llm = get_llm().bind_tools(tools)

# Модель сама решает, вызывать ли инструмент
response = llm.invoke([HumanMessage(content="Сколько будет 17 * 23?")])

print(f"Тип ответа: {type(response).__name__}")
print(f"Текст ответа: {response.content}")
print(f"tool_calls: {response.tool_calls}")

Тип ответа: AIMessage
Текст ответа: 
tool_calls: [{'name': 'multiply', 'args': {'a': 17, 'b': 23}, 'id': 'call_BaO30w8WUdVK8YX6uMLgCv9c', 'type': 'tool_call'}]


## AgentExecutor — старый подход (LangChain)

In [4]:
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Ты — полезный ассистент."),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

llm = get_llm()
agent = create_openai_functions_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools)

# Всё внутри — чёрный ящик: цикл LLM → tool → LLM → ... скрыт
result = executor.invoke({"input": "Сколько будет 6 * 7?"})
print(f"Ответ: {result['output']}")

Ответ: 6 × 7 = 42.


## ToolNode — автоматическое выполнение инструментов

In [5]:
tool_node = ToolNode(tools)
llm = get_llm()
llm_with_tools = llm.bind_tools(tools)

def agent(state: MessagesState) -> dict:
    """Агент: вызывает LLM с привязанными инструментами."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_node("tools", tool_node)
graph.add_edge(START, "agent")
graph.add_edge("agent", "tools")
graph.add_edge("tools", END)
app = graph.compile()

result = app.invoke({"messages": [HumanMessage(content="Сколько 17*23?")]})
tool_result = result["messages"][-1]
print(f"Результат ToolNode: {tool_result.content}")

Результат ToolNode: 391.0


## ReAct-агент — ручная сборка

In [6]:
llm = get_llm().bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    """Узел агента: вызывает LLM с инструментами."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_use_tools(state: MessagesState) -> str:
    """Маршрутизатор: проверяет, запросила ли модель инструменты."""
    last_message = state["messages"][-1]
    # tool_calls — список: может быть один вызов или несколько параллельных
    if last_message.tool_calls:
        return "tools"
    return "end"

# Сборка графа
graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_use_tools,
    {
        "tools": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")  # После инструментов — обратно к агенту
app = graph.compile()

# Тест 1: простой вызов одного инструмента
print("\nТест 1 — один инструмент:")
result = app.invoke({"messages": [HumanMessage(content="Сколько будет 17 * 23?")]})
print(result["messages"][-1].content)

# Тест 2: параллельный вызов нескольких инструментов
print("\nТест 2 — параллельные инструменты:")
result = app.invoke(
    {"messages": [HumanMessage(content="Какая погода в Москве и чему равно 2^10?")]}
)
print(result["messages"][-1].content)


Тест 1 — один инструмент:


17 × 23 = 391

Тест 2 — параллельные инструменты:


В Москве сейчас **+15°C, облачно**.  
**2^10 = 1024**.


## create_agent — быстрый старт

In [7]:
app = create_agent(
    model=get_llm(),
    tools=tools,
)

result = app.invoke(
    {"messages": [("user", "Найди кто создал Python")]},
)
print(result["messages"][-1].content)

Python создал Гвидо ван Россум (Guido van Rossum).

Он начал разработку языка в конце 1980-х и впервые выпустил Python в 1991 году.


## create_agent с системным промптом

In [8]:
app = create_agent(
    model=get_llm(),
    tools=tools,
    system_prompt="Ты — аналитик. Всегда проверяй факты через поиск.",
    # Можно добавить checkpointer, interrupt_before и другие опции
)

result = app.invoke({"messages": [("user", "Когда появился Python?")]})
print(result["messages"][-1].content)

Python появился в 1991 году. Его создал Гвидо ван Россум, а первая публичная версия была выпущена именно тогда.




## create_agent — response_format и context_schema

In [9]:
from pydantic import BaseModel, Field

# --- response_format: структурированный вывод ---

class WeatherReport(BaseModel):
    """Структура ответа о погоде."""

    city: str = Field(description="Название города")
    temperature: str = Field(description="Температура")
    summary: str = Field(description="Краткое описание погоды")

agent_structured = create_agent(
    # Примечание: модель также можно передать строкой, например:
    # model="openai:gpt-5.4-mini"
    model=get_llm(),  # при работе через OpenRouter модель передаётся как объект ChatOpenAI.
    tools=[get_weather],
    system_prompt="Ты — погодный ассистент. Отвечай кратко и по делу.",
    response_format=WeatherReport,
)

result = agent_structured.invoke({"messages": [("user", "Какая погода в Москве?")]})
# При response_format результат содержит structured_response
structured = result.get("structured_response")
if structured:
    print(f"Город: {structured.city}")
    print(f"Температура: {structured.temperature}")
    print(f"Описание: {structured.summary}")
else:
    print(f"Ответ: {result['messages'][-1].content}")

# --- context_schema: runtime-контекст ---
# Агент получает доступ к данным пользователя, настройкам языка и другим параметрам,
# не засоряя историю сообщений.

class UserContext(TypedDict):
    user_name: str
    language: str

agent_with_context = create_agent(
    model=get_llm(),
    tools=tools,
    system_prompt="Ты — помощник. Обращайся к пользователю по имени.",
    context_schema=UserContext,
)

result = agent_with_context.invoke(
    {"messages": [("user", "Привет, кто ты?")]},
    config={"configurable": {"user_name": "Алексей", "language": "ru"}},
)


print(f"\nОтвет с контекстом: {result['messages'][-1].content}")

Город: Москва
Температура: +15°C
Описание: облачно



Ответ с контекстом: Привет! Я помощник на базе ИИ.  
Как я могу к тебе обращаться?


## Command — маршрутизация изнутри узла

In [10]:
class RouterState(TypedDict):
    query: str
    category: str
    answer: str


def classifier_node(
    state: RouterState
) -> Command[Literal["technical", "general", "escalate"]]:
    """Классифицирует запрос и СРАЗУ направляет в нужный узел."""
    query = state["query"].lower()

    if any(word in query for word in ["код", "баг", "ошибка", "python", "api"]):
        return Command(
            update={"category": "technical"},
            goto="technical",
        )
    elif any(word in query for word in ["жалоба", "менеджер", "возврат"]):
        return Command(
            update={"category": "escalation"},
            goto="escalate",
        )
    else:
        return Command(
            update={"category": "general"},
            goto="general",
        )

def technical_node(state: RouterState) -> dict:
    return {"answer": f"Техподдержка: обработка запроса '{state['query']}'"}

def general_node(state: RouterState) -> dict:
    return {"answer": f"Общий ответ на: '{state['query']}'"}

def escalation_node(state: RouterState) -> dict:
    return {"answer": f"Эскалация: '{state['query']}' передан менеджеру"}

# Сборка графа — НЕТ add_conditional_edges, маршрутизация через Command
graph = StateGraph(RouterState)
graph.add_node("classify", classifier_node)
graph.add_node("technical", technical_node)
graph.add_node("general", general_node)
graph.add_node("escalate", escalation_node)

graph.add_edge(START, "classify")
graph.add_edge("technical", END)
graph.add_edge("general", END)
graph.add_edge("escalate", END)

app = graph.compile()

# Тест: технический запрос
result = app.invoke({"query": "У меня баг в API авторизации"})
print(f"Категория: {result['category']}")
print(f"Ответ: {result['answer']}")

# Тест: запрос на эскалацию
result = app.invoke({"query": "Хочу жалобу на обслуживание"})
print(f"\nКатегория: {result['category']}")
print(f"Ответ: {result['answer']}")

Категория: technical
Ответ: Техподдержка: обработка запроса 'У меня баг в API авторизации'

Категория: general
Ответ: Общий ответ на: 'Хочу жалобу на обслуживание'


## Streaming — потоковая обработка

In [11]:
llm = get_llm().bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_use_tools(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_use_tools,
    {
        "tools": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")
app = graph.compile()

input_messages = [("user", "Какая погода в Париже?")]
for chunk in app.stream({"messages": input_messages}, stream_mode="values"):
    # chunk — полное состояние графа
    messages = chunk["messages"]
    print(f"Сообщений: {len(messages)}, последнее: {messages[-1].type}")

Сообщений: 1, последнее: human


Сообщений: 2, последнее: ai
Сообщений: 3, последнее: tool


Сообщений: 4, последнее: ai


## Streaming — режим 'updates'

In [12]:
llm = get_llm().bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_use_tools(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_use_tools,
    {
        "tools": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")
app = graph.compile()

input_messages = [("user", "Сколько будет 17 * 23 и какая погода в Москве?")]

for chunk in app.stream({"messages": input_messages}):
    # chunk — словарь {имя_узла: обновление}
    for node_name, update in chunk.items():
        print(f"\n--- {node_name} ---")
        if "messages" in update:
            for msg in update["messages"]:
                msg_preview = msg.content[:80] if msg.content else "[tool_calls]"
                print(f"  [{msg.type}] {msg_preview}")


--- agent ---
  [ai] [tool_calls]

--- tools ---
  [tool] 391.0
  [tool] В Москва сейчас +15°C, облачно.



--- agent ---
  [ai] 17 × 23 = 391.

В Москве сейчас +15°C, облачно.


## Streaming — режим 'messages'

In [13]:
llm = get_llm().bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_use_tools(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_use_tools,
    {
        "tools": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")
app = graph.compile()

input_messages = [("user", "Расскажи коротко о квантовых компьютерах")]

for message, metadata in app.stream(
    {"messages": input_messages}, stream_mode="messages"
):
    # message — один токен (или фрагмент) сообщения
    # metadata — откуда пришёл (какой узел, какой тип)
    if message.content and metadata["langgraph_node"] == "agent":
        print(message.content, end="", flush=True)
print()

К

в

ант

овые

 компьют

...

 словами

 на

 прим

ере

.

## Streaming — режим 'custom'

In [14]:
from langgraph.config import get_stream_writer

def research_node(state: MessagesState) -> dict:
    writer = get_stream_writer()

    writer({"status": "Начинаю поиск..."})
    # В реальности здесь был бы вызов поисковой функции
    results = ["result1", "result2", "result3"]

    writer({"status": f"Найдено {len(results)} результатов, анализирую..."})
    # В реальности — анализ результатов
    analysis = "Результат анализа найденных данных."

    writer({"status": "Формирую ответ"})
    return {"messages": [AIMessage(content=analysis)]}

graph = StateGraph(MessagesState)
graph.add_node("research", research_node)
graph.add_edge(START, "research")
graph.add_edge("research", END)
app = graph.compile()

input_messages = [("user", "Исследуй тему")]

for chunk in app.stream({"messages": input_messages}, stream_mode="custom"):
    if "status" in chunk:
        print(f"Статус: {chunk['status']}")

Статус: Начинаю поиск...
Статус: Найдено 3 результатов, анализирую...
Статус: Формирую ответ


## Streaming — множественные режимы

In [15]:
llm = get_llm().bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_use_tools(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_use_tools,
    {
        "tools": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")
app = graph.compile()

input_messages = [("user", "Вопрос")]

for event in app.stream(
    {"messages": input_messages}, stream_mode=["updates", "messages"]
):
    mode = event[0]  # "updates" или "messages"
    data = event[1]  # содержимое
    print(f"[{mode}] {str(data)[:100]}")

[messages] (AIMessageChunk(content='Кон', additional_kwargs={}, response_metadata={'model_provider': 'openai'},
[messages] (AIMessageChunk(content='ечно', additional_kwargs={}, response_metadata={'model_provider': 'openai'}
[messages] (AIMessageChunk(content=' —', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, 
[messages] (AIMessageChunk(content=' зада', additional_kwargs={}, response_metadata={'model_provider': 'openai'
[messages] (AIMessageChunk(content='вайте', additional_kwargs={}, response_metadata={'model_provider': 'openai'
[messages] (AIMessageChunk(content=' вопрос', additional_kwargs={}, response_metadata={'model_provider': 'opena
[messages] (AIMessageChunk(content='.', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, i
[messages] (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model
[messages] (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop',

## Асинхронный streaming (FastAPI шаблон)

In [16]:
# Создаём агента для демонстрации
llm = get_llm().bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_use_tools(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent", should_use_tools, {"tools": "tools", "end": END}
)
graph.add_edge("tools", "agent")
app = graph.compile()

# --- Шаблон FastAPI-эндпоинта ---
# from fastapi import FastAPI
# from fastapi.responses import StreamingResponse
#
# fast_app = FastAPI()
#
# @fast_app.post("/chat")
# async def chat(question: str):
#     async def event_generator():
#         async for chunk in app.astream(
#             {"messages": [("user", question)]},
#             stream_mode="messages"
#         ):
#             message, metadata = chunk
#             if message.content and metadata["langgraph_node"] == "agent":
#                 yield f"data: {json.dumps({'token': message.content})}\n\n"
#         yield "data: [DONE]\n\n"
#
#     return StreamingResponse(event_generator(), media_type="text/event-stream")

print("Шаблон FastAPI SSE-эндпоинта — см. закомментированный код в функции.")
print("Для запуска: uvicorn module:fast_app --reload")

Шаблон FastAPI SSE-эндпоинта — см. закомментированный код в функции.
Для запуска: uvicorn module:fast_app --reload


## Streaming v2 — GraphOutput и StreamPart

In [17]:
app = create_agent(
    model=get_llm(),
    tools=tools,
    system_prompt="Ты — помощник. Отвечай кратко.",
)

# --- invoke с version="v2" ---
print("--- invoke version='v2' ---")
result = app.invoke(
    {"messages": [("user", "Сколько будет 6 * 7?")]},
    version="v2",
)
# result — GraphOutput, а не обычный dict
print(f"Тип результата: {type(result).__name__}")
print(f"Значение (value): {type(result.value).__name__}")
print(f"Прерывания (interrupts): {result.interrupts}")

# Доступ к данным через .value
last_msg = result.value["messages"][-1]
print(f"Ответ: {last_msg.content}")

# ВАЖНО: dict-доступ (result["messages"]) deprecated — используйте result.value
# result["messages"]  # DeprecationWarning, удалят в v3.0

# --- stream с version="v2" ---
print("\n--- stream version='v2' ---")
for chunk in app.stream(
    {"messages": [("user", "Какая погода в Париже?")]},
    stream_mode="values",
    version="v2",
):
    # chunk — типизированный StreamPart-словарь
    # Ключи: type, ns, data, interrupts
    print(f"  type={chunk['type']}, data_keys={list(chunk['data'].keys())}")
    if chunk.get("interrupts"):
        print(f"  interrupts={chunk['interrupts']}")

--- invoke version='v2' ---


Тип результата: GraphOutput
Значение (value): dict
Прерывания (interrupts): ()
Ответ: 42

--- stream version='v2' ---
  type=values, data_keys=['messages']


  type=values, data_keys=['messages']
  type=values, data_keys=['messages']


  type=values, data_keys=['messages']


## Продвинутые паттерны с инструментами

In [18]:
def custom_tool_node(state: MessagesState) -> dict:
    """Кастомный узел инструментов с предобработкой."""
    last_message = state["messages"][-1]
    results = []

    for call in last_message.tool_calls:
        # Логируем
        print(f"  Вызов: {call['name']}({call['args']})")

        # Находим и вызываем инструмент
        tool_fn = {t.name: t for t in tools}[call["name"]]

        try:
            result = tool_fn.invoke(call["args"])
            results.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"],
                )
            )
        except Exception as e:
            # Ошибка инструмента — не крашим агента, а сообщаем модели
            results.append(
                ToolMessage(
                    content=f"Ошибка: {str(e)}. Попробуй другой подход.",
                    tool_call_id=call["id"],
                )
            )

    return {"messages": results}

# Демонстрация: LLM генерирует tool_call, кастомный узел его обрабатывает
llm = get_llm()
llm_with_tools = llm.bind_tools(tools)
ai_message = llm_with_tools.invoke("Умножь 6 на 7")
result = custom_tool_node({"messages": [ai_message]})
print(f"Результат: {result['messages'][0].content}")

  Вызов: multiply({'a': 6, 'b': 7})
Результат: 42.0


## recursion_limit — защита от зацикливания

In [19]:
app = create_agent(
    model=get_llm(),
    tools=tools,
)

result = app.invoke(
    {"messages": [("user", "Какая погода в Москве?")]},
    config={"recursion_limit": 15},  # Максимум 15 шагов (узлов)
)
print(result["messages"][-1].content)

В Москве сейчас **+15°C, облачно**.


## Динамическая фильтрация инструментов

In [20]:
def agent_node(state: MessagesState) -> dict:
    """Агент с динамическим набором инструментов."""
    # Определяем контекст
    last_msg = state["messages"][-1].content.lower()

    if "погода" in last_msg or "температура" in last_msg:
        active_tools = [get_weather]
    elif "посчитай" in last_msg or "вычисли" in last_msg:
        active_tools = [multiply]
    else:
        active_tools = tools  # Все инструменты

    # Привязываем только релевантные инструменты
    model_with_tools = get_llm().bind_tools(active_tools)
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_use_tools(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_use_tools,
    {
        "tools": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")
app = graph.compile()

result = app.invoke({"messages": [HumanMessage(content="Какая погода в Лондоне?")]})
print(result["messages"][-1].content)

В Лондоне сейчас +15°C, облачно.
